In [2]:
from pathlib import Path
import pandas as pd

# Locate the project root
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

candidate_path = (
    project_root
    / "data"
    / "interim"
    / "high_confidence_chinese_candidates.csv"
)

candidates = pd.read_csv(
    candidate_path,
    dtype="string"
)

print("Candidate rows:", len(candidates))

print(
    "Duplicate facility IDs:",
    candidates["facility_id"].duplicated().sum()
)

print("\nColumns:")
print(candidates.columns.tolist())

address_columns = [
    "facility_address",
    "facility_city",
    "facility_state",
    "facility_zip"
]

missing_address_summary = (
    candidates[address_columns]
    .isna()
    .sum()
    .rename_axis("column")
    .reset_index(name="missing_count")
)

print("\nMissing address components:")
display(missing_address_summary)

display(
    candidates[
        [
            "facility_id",
            "facility_name",
            "facility_address",
            "facility_city",
            "facility_state",
            "facility_zip"
        ]
    ].head()
)

Candidate rows: 559
Duplicate facility IDs: 0

Columns:
['facility_id', 'facility_name', 'facility_address', 'facility_city', 'facility_state', 'facility_zip', 'latest_inspection_date', 'program_count', 'owner_count', 'average_latest_score', 'minimum_latest_score', 'maximum_latest_score', 'special_venue', 'name_normalized', 'chinese_strong_match', 'chinese_review_match', 'other_cuisine_signal', 'chinese_candidate_status', 'matched_strong_keywords', 'matched_review_keywords', 'matched_other_cuisine_keywords']

Missing address components:


,column,missing_count
0,facility_address,0
1,facility_city,0
2,facility_state,0
3,facility_zip,0


,facility_id,facility_name,facility_address,facility_city,facility_state,facility_zip
0,FA0004830,AA CHINESE EXPRESS FAST FOOD,5101 S AVALON BLVD,LOS ANGELES,CA,90011
1,FA0005102,ACC CHINESE FAST FOOD,38 S PALM AVE,ALHAMBRA,CA,91801
2,FA0005216,BEN'S CHINESE RESTAURANT,10262 ROSECRANS AVE,BELLFLOWER,CA,90706
3,FA0008150,BAMBOO GARDEN CHINESE REST,2632 W VALLEY BLVD,ALHAMBRA,CA,91803
4,FA0008745,CHINA BEAUTY,5465 N FIGUEROA ST,LOS ANGELES,CA,90042


In [3]:
# Prepare addresses for the Census batch geocoder

geocoding_input = candidates[
    [
        "facility_id",
        "facility_address",
        "facility_city",
        "facility_state",
        "facility_zip"
    ]
].copy()

# Preserve the original address and create a geocoding version
geocoding_input["original_address"] = (
    geocoding_input["facility_address"]
)

# Remove suite, unit, and secondary-address information
geocoding_input["facility_address"] = (
    geocoding_input["facility_address"]
    .str.upper()
    .str.strip()
    .str.replace(
        r"\s+(?:#|STE|SUITE|UNIT|BLDG)\s*.*$",
        "",
        regex=True
    )
)

geocoding_input["facility_city"] = (
    geocoding_input["facility_city"]
    .str.upper()
    .str.strip()
)

geocoding_input["facility_state"] = (
    geocoding_input["facility_state"]
    .str.upper()
    .str.strip()
)

# Keep only the five-digit ZIP code
geocoding_input["facility_zip"] = (
    geocoding_input["facility_zip"]
    .str.extract(r"(\d{5})", expand=False)
)

batch_columns = [
    "facility_id",
    "facility_address",
    "facility_city",
    "facility_state",
    "facility_zip"
]

batch_file = geocoding_input[batch_columns].copy()

geocoding_dir = (
    project_root
    / "data"
    / "interim"
    / "geocoding"
)

geocoding_dir.mkdir(
    parents=True,
    exist_ok=True
)

batch_input_path = (
    geocoding_dir
    / "census_geocoder_chinese_candidates_input.csv"
)

# Census batch files must not contain a header row
batch_file.to_csv(
    batch_input_path,
    index=False,
    header=False
)

print("Batch rows prepared:", len(batch_file))
print("Missing values:", batch_file.isna().sum().sum())
print("Batch file saved to:", batch_input_path)

display(
    geocoding_input[
        [
            "facility_id",
            "original_address",
            "facility_address",
            "facility_city",
            "facility_state",
            "facility_zip"
        ]
    ].head(10)
)

Batch rows prepared: 559
Missing values: 1
Batch file saved to: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\interim\geocoding\census_geocoder_chinese_candidates_input.csv


,facility_id,original_address,facility_address,facility_city,facility_state,facility_zip
0,FA0004830,5101 S AVALON BLVD,5101 S AVALON BLVD,LOS ANGELES,CA,90011
1,FA0005102,38 S PALM AVE,38 S PALM AVE,ALHAMBRA,CA,91801
2,FA0005216,10262 ROSECRANS AVE,10262 ROSECRANS AVE,BELLFLOWER,CA,90706
3,FA0008150,2632 W VALLEY BLVD,2632 W VALLEY BLVD,ALHAMBRA,CA,91803
4,FA0008745,5465 N FIGUEROA ST,5465 N FIGUEROA ST,LOS ANGELES,CA,90042
5,FA0008761,855 W VICTORIA ST STE B-1,855 W VICTORIA ST,COMPTON,CA,90220
6,FA0008762,11845 BRADDOCK DR,11845 BRADDOCK DR,CULVER CITY,CA,90230
7,FA0008768,14365 S CLARK AVE,14365 S CLARK AVE,BELLFLOWER,CA,90706
8,FA0008776,5424 LAUREL CANYON BLVD STE C,5424 LAUREL CANYON BLVD,VALLEY VILLAGE,CA,91607
9,FA0008781,7116 EASTERN AVE,7116 EASTERN AVE,BELL GARDENS,CA,90201


In [4]:
# Identify the row with a missing batch component

missing_batch_rows = geocoding_input.loc[
    geocoding_input[batch_columns].isna().any(axis=1),
    [
        "facility_id",
        "original_address",
        "facility_address",
        "facility_city",
        "facility_state",
        "facility_zip"
    ]
]

display(missing_batch_rows)

# Check the original ZIP value from the candidate table
problem_facility_ids = missing_batch_rows["facility_id"]

display(
    candidates.loc[
        candidates["facility_id"].isin(problem_facility_ids),
        [
            "facility_id",
            "facility_name",
            "facility_address",
            "facility_city",
            "facility_state",
            "facility_zip"
        ]
    ]
)

,facility_id,original_address,facility_address,facility_city,facility_state,facility_zip
537,FA0365049,815 W NAOMI AVE #E,815 W NAOMI AVE,ARCADIA,CA,<NA>


,facility_id,facility_name,facility_address,facility_city,facility_state,facility_zip
537,FA0365049,YUNXIANG SZECHUAN RESTAURANT,815 W NAOMI AVE #E,ARCADIA,CA,9100-


In [5]:
# Documented correction for an incomplete source ZIP code

zip_corrections = {
    "FA0365049": "91007"
}

geocoding_input["zip_corrected"] = False

for facility_id, corrected_zip in zip_corrections.items():
    correction_mask = (
        geocoding_input["facility_id"] == facility_id
    )

    geocoding_input.loc[
        correction_mask,
        "facility_zip"
    ] = corrected_zip

    geocoding_input.loc[
        correction_mask,
        "zip_corrected"
    ] = True

# Rebuild and overwrite the batch file
batch_file = geocoding_input[batch_columns].copy()

batch_file.to_csv(
    batch_input_path,
    index=False,
    header=False
)

print("Batch rows:", len(batch_file))
print("Missing values:", batch_file.isna().sum().sum())
print(
    "ZIP corrections:",
    geocoding_input["zip_corrected"].sum()
)

display(
    geocoding_input.loc[
        geocoding_input["zip_corrected"],
        [
            "facility_id",
            "original_address",
            "facility_address",
            "facility_city",
            "facility_state",
            "facility_zip",
            "zip_corrected"
        ]
    ]
)

Batch rows: 559
Missing values: 0
ZIP corrections: 1


,facility_id,original_address,facility_address,facility_city,facility_state,facility_zip,zip_corrected
537,FA0365049,815 W NAOMI AVE #E,815 W NAOMI AVE,ARCADIA,CA,91007,True


In [6]:
import requests

# Submit the batch file to the Census Geocoder

geocoder_url = (
    "https://geocoding.geo.census.gov/"
    "geocoder/locations/addressbatch"
)

batch_output_path = (
    geocoding_dir
    / "census_geocoder_chinese_candidates_output.csv"
)

with open(batch_input_path, "rb") as address_file:
    response = requests.post(
        geocoder_url,
        files={
            "addressFile": (
                batch_input_path.name,
                address_file,
                "text/csv"
            )
        },
        data={
            "benchmark": "Public_AR_Current"
        },
        timeout=300
    )

print("Request status:", response.status_code)
print(
    "Response type:",
    response.headers.get("Content-Type")
)

response.raise_for_status()

# Check that an HTML error page was not returned
response_type = response.headers.get(
    "Content-Type",
    ""
).lower()

if "text/html" in response_type:
    raise RuntimeError(
        "The Census Geocoder returned an HTML page "
        "instead of a CSV result."
    )

batch_output_path.write_bytes(response.content)

print(
    "Response size:",
    round(len(response.content) / 1024, 2),
    "KB"
)

print("Output saved to:", batch_output_path)

# Preview the first few raw response lines
preview_lines = response.text.splitlines()[:3]

print("\nRaw output preview:")
for line in preview_lines:
    print(line)

Request status: 200
Response type: text/plain
Response size: 90.01 KB
Output saved to: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\interim\geocoding\census_geocoder_chinese_candidates_output.csv

Raw output preview:
"FA0005102","38 S PALM AVE, ALHAMBRA, CA, 91801","Match","Exact","38 S PALM AVE, ALHAMBRA, CA, 91801","-118.145949592493,34.090655199062","141635542","L"
"FA0328881","2560 LINCOLN BLVD, VENICE, CA, 90291","Match","Exact","2560 LINCOLN BLVD, VENICE, CA, 90291","-118.448380703883,33.991236218196","141577093","L"
"FA0296845","1754 S GRAND AVE, GLENDORA, CA, 91740","Match","Exact","1754 S GRAND AVE, GLENDORA, CA, 91740","-117.87243267238,34.106991178896","141724183","L"


In [7]:
# Parse the Census batch geocoder output

geocoder_columns = [
    "facility_id",
    "input_address",
    "match_status",
    "match_type",
    "matched_address",
    "coordinates",
    "tiger_line_id",
    "tiger_line_side"
]

geocoded_results = pd.read_csv(
    batch_output_path,
    header=None,
    names=geocoder_columns,
    dtype="string"
)

# Split coordinates into longitude and latitude
coordinate_parts = (
    geocoded_results["coordinates"]
    .str.split(",", n=1, expand=True)
)

geocoded_results["longitude"] = pd.to_numeric(
    coordinate_parts[0],
    errors="coerce"
)

geocoded_results["latitude"] = pd.to_numeric(
    coordinate_parts[1],
    errors="coerce"
)

print("Output rows:", len(geocoded_results))

print(
    "Duplicate facility IDs:",
    geocoded_results["facility_id"].duplicated().sum()
)

print(
    "Missing longitude:",
    geocoded_results["longitude"].isna().sum()
)

print(
    "Missing latitude:",
    geocoded_results["latitude"].isna().sum()
)

match_summary = (
    geocoded_results["match_status"]
    .value_counts(dropna=False)
    .rename_axis("match_status")
    .reset_index(name="facility_count")
)

display(match_summary)

display(
    geocoded_results[
        [
            "facility_id",
            "match_status",
            "match_type",
            "matched_address",
            "longitude",
            "latitude"
        ]
    ].head(10)
)

Output rows: 559
Duplicate facility IDs: 0
Missing longitude: 19
Missing latitude: 19


,match_status,facility_count
0,Match,540
1,No_Match,16
2,Tie,3


,facility_id,match_status,match_type,matched_address,longitude,latitude
0,FA0005102,Match,Exact,"38 S PALM AVE, ALHAMBRA, CA, 91801",-118.14595,34.090655
1,FA0328881,Match,Exact,"2560 LINCOLN BLVD, VENICE, CA, 90291",-118.448381,33.991236
2,FA0296845,Match,Exact,"1754 S GRAND AVE, GLENDORA, CA, 91740",-117.872433,34.106991
3,FA0305763,Match,Exact,"519 W MANCHESTER AVE, LOS ANGELES, CA, 90044",-118.283626,33.96013
4,FA0043739,Match,Exact,"811 PLAZA DR, WEST COVINA, CA, 91790",-117.927464,34.069895
5,FA0043738,No_Match,<NA>,<NA>,<NA>,<NA>
6,FA0043737,Match,Exact,"7000 EASTERN AVE, BELL GARDENS, CA, 90201",-118.164321,33.969783
7,FA0352514,Match,Non_Exact,"11058 SANTA MONICA BLVD, LOS ANGELES, CA, 90025",-118.443343,34.047899
8,FA0037296,Match,Exact,"1936 S PACIFIC AVE, SAN PEDRO, CA, 90731",-118.287893,33.726615
9,FA0279151,Match,Exact,"17200 VENTURA BLVD, ENCINO, CA, 91316",-118.507656,34.160491


In [8]:
import geopandas as gpd

# Attach geocoding results to the original candidate records

candidates_geocoded = candidates.merge(
    geocoded_results,
    on="facility_id",
    how="left",
    validate="one_to_one"
)

print(
    "Candidate rows after merge:",
    len(candidates_geocoded)
)

# Keep records with usable coordinates
matched_candidates = candidates_geocoded.loc[
    candidates_geocoded["longitude"].notna()
    & candidates_geocoded["latitude"].notna()
].copy()

# Convert longitude and latitude into point geometry
candidate_points = gpd.GeoDataFrame(
    matched_candidates,
    geometry=gpd.points_from_xy(
        matched_candidates["longitude"],
        matched_candidates["latitude"]
    ),
    crs="EPSG:4326"
)

# Load the tract polygons created earlier
tract_geopackage_path = (
    project_root
    / "data"
    / "processed"
    / "la_tract_demographics_2024.gpkg"
)

tracts = gpd.read_file(
    tract_geopackage_path,
    layer="la_tract_demographics"
)

# Match the point coordinate system to the tract layer
candidate_points = candidate_points.to_crs(
    tracts.crs
)

# Assign each restaurant point to a Census tract
candidate_points_with_tract = gpd.sjoin(
    candidate_points,
    tracts[
        [
            "GEOID",
            "chinese_share_pct",
            "chinese_total_estimate",
            "median_household_income",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)

print("Geocoded candidate points:", len(candidate_points))

print(
    "Points inside an LA County tract:",
    candidate_points_with_tract["GEOID"]
    .notna()
    .sum()
)

print(
    "Points outside LA County tracts:",
    candidate_points_with_tract["GEOID"]
    .isna()
    .sum()
)

display(
    candidate_points_with_tract.loc[
        candidate_points_with_tract["GEOID"].isna(),
        [
            "facility_id",
            "facility_name",
            "facility_address",
            "facility_city",
            "matched_address",
            "longitude",
            "latitude"
        ]
    ]
)

Candidate rows after merge: 559
Geocoded candidate points: 540
Points inside an LA County tract: 540
Points outside LA County tracts: 0


,facility_id,facility_name,facility_address,facility_city,matched_address,longitude,latitude


In [9]:
# Add geocoding metadata

candidate_points_with_tract["geocode_source"] = (
    "US Census Batch Geocoder"
)

candidate_points_with_tract["geocode_review_status"] = (
    "initial_match"
)

# Remove the temporary spatial-join index
candidate_points_export = (
    candidate_points_with_tract
    .drop(columns=["index_right"], errors="ignore")
    .copy()
)

print(
    "Duplicate facility IDs in spatial output:",
    candidate_points_export["facility_id"]
    .duplicated()
    .sum()
)

# Check for field-name conflicts before GeoPackage export
field_names_lower = [
    column.lower()
    for column in candidate_points_export.columns
]

print(
    "Case-insensitive duplicate fields:",
    len(field_names_lower) - len(set(field_names_lower))
)

# Save the successfully geocoded restaurant points
restaurant_geopackage_path = (
    project_root
    / "data"
    / "processed"
    / "chinese_restaurant_candidates_geocoded.gpkg"
)

if restaurant_geopackage_path.exists():
    restaurant_geopackage_path.unlink()

candidate_points_export.to_file(
    restaurant_geopackage_path,
    layer="chinese_restaurant_candidates",
    driver="GPKG",
    index=False
)

# Save unmatched and tied addresses for later review
unmatched_candidates = candidates_geocoded.loc[
    candidates_geocoded["longitude"].isna()
    | candidates_geocoded["latitude"].isna()
].copy()

unmatched_output_path = (
    geocoding_dir
    / "chinese_candidates_geocoding_review.csv"
)

unmatched_candidates.to_csv(
    unmatched_output_path,
    index=False
)

print("\nSpatial restaurant rows saved:", len(candidate_points_export))
print("Addresses requiring review:", len(unmatched_candidates))
print("Restaurant GeoPackage:", restaurant_geopackage_path)
print("Review CSV:", unmatched_output_path)

Duplicate facility IDs in spatial output: 0
Case-insensitive duplicate fields: 0

Spatial restaurant rows saved: 540
Addresses requiring review: 19
Restaurant GeoPackage: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\processed\chinese_restaurant_candidates_geocoded.gpkg
Review CSV: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\interim\geocoding\chinese_candidates_geocoding_review.csv


In [10]:
# Review Census geocoding match quality

match_type_summary = (
    candidate_points_export["match_type"]
    .value_counts(dropna=False)
    .rename_axis("match_type")
    .reset_index(name="facility_count")
)

display(match_type_summary)

exact_match_count = (
    candidate_points_export["match_type"] == "Exact"
).sum()

exact_match_rate = (
    exact_match_count
    / len(candidate_points_export)
    * 100
)

print(
    "Exact-match rate among geocoded points:",
    round(exact_match_rate, 2),
    "%"
)

non_exact_review = candidate_points_export.loc[
    candidate_points_export["match_type"] != "Exact",
    [
        "facility_id",
        "facility_name",
        "facility_address",
        "facility_city",
        "facility_zip",
        "input_address",
        "matched_address",
        "longitude",
        "latitude"
    ]
].copy()

print(
    "Non-exact matches requiring review:",
    len(non_exact_review)
)

display(non_exact_review.head(20))

,match_type,facility_count
0,Exact,422
1,Non_Exact,118


Exact-match rate among geocoded points: 78.15 %
Non-exact matches requiring review: 118


,facility_id,facility_name,facility_address,facility_city,facility_zip,input_address,matched_address,longitude,latitude
0,FA0004830,AA CHINESE EXPRESS FAST FOOD,5101 S AVALON BLVD,LOS ANGELES,90011,"5101 S AVALON BLVD, LOS ANGELES, CA, 90011","5101 AVALON BLVD, LOS ANGELES, CA, 90011",-118.265277,33.99652
7,FA0008768,CHINA EXPRESS,14365 S CLARK AVE,BELLFLOWER,90706,"14365 S CLARK AVE, BELLFLOWER, CA, 90706","14365 CLARK AVE, BELLFLOWER, CA, 90706",-118.134117,33.902518
13,FA0009172,BEST CHINA KITCHEN,17234 S DOWNEY AVE,BELLFLOWER,90706,"17234 S DOWNEY AVE, BELLFLOWER, CA, 90706","17234 DOWNEY AVE, BELLFLOWER, CA, 90706",-118.151226,33.875895
17,FA0014616,CHINA WOK EXPRESS,11745 E WHITTIER BLVD,WHITTIER,90601,"11745 E WHITTIER BLVD, WHITTIER, CA, 90601","11745 WHITTIER BLVD, WHITTIER, CA, 90601",-118.051962,33.981932
20,FA0014860,CHINESE EXPRESS,1655 E 103RD ST,LOS ANGELES,90022,"1655 E 103RD ST, LOS ANGELES, CA, 90022","1655 E 103RD ST, LOS ANGELES, CA, 90002",-118.244732,33.94326
27,FA0017327,HONG KONG BAKERY & DELI,21720 VERMONT AVE,TORRANCE,90502,"21720 VERMONT AVE, TORRANCE, CA, 90502","21720 S VERMONT AVE, TORRANCE, CA, 90502",-118.290231,33.831256
43,FA0027625,PANDA EXPRESS #1372,13471 E TELEGRAPH RD STE #C,WHITTIER,90605,"13471 E TELEGRAPH RD, WHITTIER, CA, 90605","13471 TELEGRAPH RD, WHITTIER, CA, 90605",-118.044012,33.938754
45,FA0027627,PANDA EXPRESS #1056,150 CITADEL DR # #B,COMMERCE,91770,"150 CITADEL DR, COMMERCE, CA, 91770","150 CITADEL DR, COMMERCE, CA, 90040",-118.152807,34.004886
47,FA0027629,PANDA EXPRESS #351,1632 MONTEBELLO TOWN CENTER DR,MONTEBELLO,90640,"1632 MONTEBELLO TOWN CENTER DR, MONTEBELLO, CA...","1632 MONTEBELLO TOWN CENTER, MONTEBELLO, CA, 9...",-118.086458,34.036988
59,FA0027641,PANDA EXPRESS #982,417 N PACIFIC COAST HWY # G105,REDONDO BEACH,90277,"417 N PACIFIC COAST HWY, REDONDO BEACH, CA, 90277","417 S PACIFIC COAST HWY, REDONDO BEACH, CA, 90277",-118.385314,33.836167


In [11]:
# Parse input and matched address components

input_parts = candidate_points_export["input_address"].str.extract(
    r"^(.*),\s*([^,]+),\s*([A-Z]{2}),\s*(\d{5})$"
)

input_parts.columns = [
    "input_street",
    "input_city",
    "input_state",
    "input_zip"
]

matched_parts = candidate_points_export["matched_address"].str.extract(
    r"^(.*),\s*([^,]+),\s*([A-Z]{2}),\s*(\d{5})$"
)

matched_parts.columns = [
    "matched_street",
    "matched_city",
    "matched_state",
    "matched_zip"
]

candidate_points_export = pd.concat(
    [
        candidate_points_export,
        input_parts,
        matched_parts
    ],
    axis=1
)

# Extract and compare house numbers
candidate_points_export["input_house_number"] = (
    candidate_points_export["input_street"]
    .str.extract(
        r"^\s*(\d+(?:-\d+)?(?:\s+1/2)?)",
        expand=False
    )
)

candidate_points_export["matched_house_number"] = (
    candidate_points_export["matched_street"]
    .str.extract(
        r"^\s*(\d+(?:-\d+)?(?:\s+1/2)?)",
        expand=False
    )
)

candidate_points_export["house_number_match"] = (
    candidate_points_export["input_house_number"]
    .eq(candidate_points_export["matched_house_number"])
    .fillna(False)
)

candidate_points_export["zip_match"] = (
    candidate_points_export["input_zip"]
    .eq(candidate_points_export["matched_zip"])
    .fillna(False)
)

candidate_points_export["city_match"] = (
    candidate_points_export["input_city"]
    .eq(candidate_points_export["matched_city"])
    .fillna(False)
)

# Detect cases where both addresses contain conflicting directions
candidate_points_export["input_direction"] = (
    candidate_points_export["input_street"]
    .str.extract(
        r"^\s*\d+(?:-\d+)?(?:\s+1/2)?\s+([NSEW])\b",
        expand=False
    )
)

candidate_points_export["matched_direction"] = (
    candidate_points_export["matched_street"]
    .str.extract(
        r"^\s*\d+(?:-\d+)?(?:\s+1/2)?\s+([NSEW])\b",
        expand=False
    )
)

candidate_points_export["direction_conflict"] = (
    candidate_points_export["input_direction"].notna()
    & candidate_points_export["matched_direction"].notna()
    & (
        candidate_points_export["input_direction"]
        != candidate_points_export["matched_direction"]
    )
)

# Assign a preliminary quality category
candidate_points_export["geocode_quality"] = "manual_review"

candidate_points_export.loc[
    candidate_points_export["match_type"] == "Exact",
    "geocode_quality"
] = "accepted_exact"

candidate_points_export.loc[
    (candidate_points_export["match_type"] == "Non_Exact")
    & candidate_points_export["house_number_match"]
    & candidate_points_export["zip_match"]
    & candidate_points_export["city_match"]
    & ~candidate_points_export["direction_conflict"],
    "geocode_quality"
] = "accepted_standardization"

quality_summary = (
    candidate_points_export["geocode_quality"]
    .value_counts()
    .rename_axis("geocode_quality")
    .reset_index(name="facility_count")
)

display(quality_summary)

manual_review_matches = candidate_points_export.loc[
    candidate_points_export["geocode_quality"] == "manual_review",
    [
        "facility_id",
        "facility_name",
        "input_address",
        "matched_address",
        "house_number_match",
        "zip_match",
        "city_match",
        "direction_conflict",
        "longitude",
        "latitude"
    ]
].copy()

print(
    "Matched addresses needing manual review:",
    len(manual_review_matches)
)

display(manual_review_matches.head(30))

,geocode_quality,facility_count
0,accepted_exact,422
1,accepted_standardization,88
2,manual_review,30


Matched addresses needing manual review: 30


,facility_id,facility_name,input_address,matched_address,house_number_match,zip_match,city_match,direction_conflict,longitude,latitude
20,FA0014860,CHINESE EXPRESS,"1655 E 103RD ST, LOS ANGELES, CA, 90022","1655 E 103RD ST, LOS ANGELES, CA, 90002",True,False,True,False,-118.244732,33.94326
45,FA0027627,PANDA EXPRESS #1056,"150 CITADEL DR, COMMERCE, CA, 91770","150 CITADEL DR, COMMERCE, CA, 90040",True,False,True,False,-118.152807,34.004886
59,FA0027641,PANDA EXPRESS #982,"417 N PACIFIC COAST HWY, REDONDO BEACH, CA, 90277","417 S PACIFIC COAST HWY, REDONDO BEACH, CA, 90277",True,True,True,True,-118.385314,33.836167
66,FA0027649,PANDA EXPRESS,"856 E ALOSTA AVE, AZUSA, CA, 91702","856 W ALOSTA AVE, AZUSA, CA, 91702",True,True,True,True,-117.89151,34.128965
88,FA0043075,PANDA EXPRESS #647,"1541 N VICTORY PL, BURBANK, CA, 91506","1541 N VICTORY PL, BURBANK, CA, 91502",True,False,True,False,-118.325984,34.189677
102,FA0043093,PANDA EXPRESS #792,"2919 LOS FELIZ BLVD, ACTON, CA, 90039","2919 LOS FELIZ BLVD, LOS ANGELES, CA, 90039",True,True,False,False,-118.263986,34.125948
105,FA0043097,PANDA EXPRESS,"39450 E 10TH ST W, PALMDALE, CA, 93551","39450 10TH ST E, PALMDALE, CA, 93550",True,False,True,False,-118.111977,34.599281
141,FA0043740,PANDA EXPRESS #1170,"8510 FIRESTONE BLVD, DOWNEY, CA, 90240","8510 FIRESTONE BLVD, DOWNEY, CA, 90241",True,False,True,False,-118.129367,33.938128
163,FA0066116,DIM SUM HOUSE,"500 W MAIN ST ABC, ALHAMBRA, CA, 91801","500 E MAIN ST, ALHAMBRA, CA, 91801",True,True,True,True,-118.121724,34.09752
176,FA0068243,PANDA EXPRESS #1473,"18111 NORDHOFF ST, NORTHRIDGE, CA, 91330","18111 NORDHOFF ST, NORTHRIDGE, CA, 91325",True,False,True,False,-118.527629,34.235548


In [12]:
# Save the updated point layer with geocoding quality fields

if restaurant_geopackage_path.exists():
    restaurant_geopackage_path.unlink()

candidate_points_export.to_file(
    restaurant_geopackage_path,
    layer="chinese_restaurant_candidates",
    driver="GPKG",
    index=False
)

manual_review_output_path = (
    geocoding_dir
    / "matched_addresses_manual_review.csv"
)

manual_review_matches.to_csv(
    manual_review_output_path,
    index=False
)

accepted_count = (
    candidate_points_export["geocode_quality"]
    .isin(
        [
            "accepted_exact",
            "accepted_standardization"
        ]
    )
    .sum()
)

print("Point rows saved:", len(candidate_points_export))
print("Accepted point rows:", accepted_count)
print(
    "Matched points requiring review:",
    len(manual_review_matches)
)
print(
    "Unmatched or tied addresses:",
    len(unmatched_candidates)
)

print("\nUpdated GeoPackage:")
print(restaurant_geopackage_path)

print("\nManual-review CSV:")
print(manual_review_output_path)

Point rows saved: 540
Accepted point rows: 510
Matched points requiring review: 30
Unmatched or tied addresses: 19

Updated GeoPackage:
c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\processed\chinese_restaurant_candidates_geocoded.gpkg

Manual-review CSV:
c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\interim\geocoding\matched_addresses_manual_review.csv


In [13]:
# Calculate competition near the center of each Census tract

accepted_quality = [
    "accepted_exact",
    "accepted_standardization"
]

accepted_points = candidate_points_export.loc[
    candidate_points_export["geocode_quality"].isin(
        accepted_quality
    )
].copy()

# Use a projected California coordinate system measured in meters
projected_crs = "EPSG:3310"

accepted_points_projected = accepted_points.to_crs(
    projected_crs
)

tract_centers = (
    tracts[
        [
            "GEOID",
            "geometry"
        ]
    ]
    .copy()
    .to_crs(projected_crs)
)

# Create one representative center point for each tract
tract_centers["geometry"] = (
    tract_centers.geometry.centroid
)

def count_restaurants_within_radius(
    center_points,
    restaurant_points,
    radius_meters,
    output_column
):
    buffers = center_points.copy()

    buffers["geometry"] = (
        buffers.geometry.buffer(radius_meters)
    )

    joined = gpd.sjoin(
        buffers,
        restaurant_points[
            [
                "facility_id",
                "geometry"
            ]
        ],
        how="left",
        predicate="contains"
    )

    counts = (
        joined.groupby("GEOID")["facility_id"]
        .count()
        .rename(output_column)
    )

    return counts

meters_per_mile = 1609.344

competition_1_mile = count_restaurants_within_radius(
    tract_centers,
    accepted_points_projected,
    1 * meters_per_mile,
    "competitors_within_1_mile"
)

competition_3_miles = count_restaurants_within_radius(
    tract_centers,
    accepted_points_projected,
    3 * meters_per_mile,
    "competitors_within_3_miles"
)

competition_metrics = (
    tract_centers[
        [
            "GEOID"
        ]
    ]
    .set_index("GEOID")
    .join(competition_1_mile)
    .join(competition_3_miles)
    .reset_index()
)

print("Tracts calculated:", len(competition_metrics))

print(
    "Maximum competitors within 1 mile:",
    competition_metrics[
        "competitors_within_1_mile"
    ].max()
)

print(
    "Maximum competitors within 3 miles:",
    competition_metrics[
        "competitors_within_3_miles"
    ].max()
)

display(
    competition_metrics.sort_values(
        "competitors_within_3_miles",
        ascending=False
    ).head(10)
)

Tracts calculated: 2498
Maximum competitors within 1 mile: 16
Maximum competitors within 3 miles: 52


,GEOID,competitors_within_1_mile,competitors_within_3_miles
184,06037213402,12,52
1073,06037221810,5,51
2280,06037224200,5,51
1103,06037224310,8,51
2281,06037224320,6,51
2245,06037224010,6,51
2385,06037481712,8,51
161,06037213401,14,50
1112,06037209810,12,50
1179,06037482202,6,50


In [14]:
# Join competition metrics to the tract demographic layer

tract_site_selection_inputs = tracts.merge(
    competition_metrics,
    on="GEOID",
    how="left",
    validate="one_to_one"
)

competition_columns = [
    "competitors_within_1_mile",
    "competitors_within_3_miles"
]

tract_site_selection_inputs[competition_columns] = (
    tract_site_selection_inputs[competition_columns]
    .fillna(0)
    .astype(int)
)

print(
    "Final tract rows:",
    len(tract_site_selection_inputs)
)

print(
    "Missing 1-mile competition values:",
    tract_site_selection_inputs[
        "competitors_within_1_mile"
    ].isna().sum()
)

print(
    "Missing 3-mile competition values:",
    tract_site_selection_inputs[
        "competitors_within_3_miles"
    ].isna().sum()
)

# Save a non-spatial CSV for SQL and Excel analysis
competition_csv_path = (
    project_root
    / "data"
    / "processed"
    / "la_tract_demographic_competition_metrics.csv"
)

tract_site_selection_inputs.drop(
    columns="geometry"
).to_csv(
    competition_csv_path,
    index=False
)

# Save the complete spatial input layer
site_selection_gpkg_path = (
    project_root
    / "data"
    / "processed"
    / "la_tract_site_selection_inputs.gpkg"
)

if site_selection_gpkg_path.exists():
    site_selection_gpkg_path.unlink()

tract_site_selection_inputs.to_file(
    site_selection_gpkg_path,
    layer="tract_site_selection_inputs",
    driver="GPKG",
    index=False
)

print("\nCSV saved to:", competition_csv_path)
print("GeoPackage saved to:", site_selection_gpkg_path)

Final tract rows: 2498
Missing 1-mile competition values: 0
Missing 3-mile competition values: 0

CSV saved to: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\processed\la_tract_demographic_competition_metrics.csv
GeoPackage saved to: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\processed\la_tract_site_selection_inputs.gpkg
